In [2]:
# Quarterly depedency graph

In [3]:
import os
os.getcwd()

'c:\\Users\\ugne.keliauskaite\\Bruegel GitLab\\research-2021-11-european-natural-gas-imports\\standalone pieces of code'

In [4]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [5]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [6]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [7]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [8]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [9]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [10]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [11]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [12]:
# to align with Bloombgerg lng
# agsi_19= agsi_19.iloc[:0] #Ugne change this one!

In [13]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2024-08-31     75448.0
2024-09-30     82988.8
2024-10-31     91937.6
2024-11-30     98245.7
2024-12-31    109031.3
Freq: M, Name: sendOut, Length: 72, dtype: float64

In [14]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [15]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2024-08-31     75448.0
2024-09-30     82988.8
2024-10-31     91937.6
2024-11-30     98245.7
2024-12-31    109031.3
Freq: M, Name: sendOut, Length: 72, dtype: float64

In [16]:
ratios_df= ratios_df.iloc[:-1] #Ugne change this one!

In [17]:
ratios_df

,Qatar,Algeria,Nigeria,Russia,Norway,United States,Trinidad & Tobago,Egypt,Other,Tot
dates,,,,,,,,,,
2019-01-31,0.227813,0.104325,0.187116,0.193431,0.046563,0.139817,0.074051,0.000000,0.026883,1.0
2019-02-28,0.195039,0.117453,0.146219,0.202086,0.062567,0.104679,0.076749,0.000000,0.095208,1.0
2019-03-31,0.217705,0.092288,0.139617,0.207924,0.054209,0.163316,0.068449,0.025039,0.031453,1.0
2019-04-30,0.174857,0.129060,0.138185,0.228259,0.052365,0.155742,0.085473,0.024848,0.011212,1.0
2019-05-31,0.220073,0.115558,0.135534,0.252179,0.054785,0.108252,0.056022,0.024447,0.033150,1.0
...,...,...,...,...,...,...,...,...,...,...
2024-08-31,0.132085,0.065166,0.063576,0.164130,0.061752,0.419936,0.058421,0.000000,0.034935,1.0
2024-09-30,0.110343,0.105233,0.043391,0.198578,0.044815,0.426727,0.045515,0.000000,0.025399,1.0
2024-10-31,0.116477,0.109085,0.055691,0.156780,0.054992,0.454564,0.000000,0.000000,0.052412,1.0


In [18]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [19]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [20]:
lng['Total less Russia and USA'] = lng['Tot'] - lng['Russia'] - lng['United States']

In [21]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2024-12-31', freq='M')

In [22]:
entsog = entsog['2019':]

In [23]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [24]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [25]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [26]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [27]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
del graph['Russia']
graph['LNG less RU and USA'] = lng_q['Total less Russia and USA']
graph = graph['2019':]

In [28]:
graph.tail()

,Norway,Algeria,UK,Azerbaijan,Libya,Ukraine Gas Transit,"Yamal (BY,PL)",Nord Stream,Turkstream,USA LNG,Russia LNG,LNG less RU and USA
dates,,,,,,,,,,,,
2023-12-31,23863.828909,8266.048489,3141.930247,3233.510731,670.741001,4261.379134,0.0,0.0,4071.679780,17030.290121,4171.789277,12172.804097
2024-03-31,24152.453088,7446.442823,1869.480192,3202.055103,481.842846,3975.994269,0.0,0.0,3903.919757,16005.540028,5973.680613,9404.585184
2024-06-30,23903.666352,8608.889462,3673.189310,3171.037865,419.845705,4112.429768,0.0,0.0,3874.656347,12858.688782,5074.692390,10407.832420
2024-09-30,21743.195581,7082.813214,5152.462673,2920.944318,212.792361,4178.388680,0.0,0.0,4442.340756,9753.678915,4805.904604,9230.969879
2024-12-31,23482.078217,8859.495027,2043.887414,3389.197710,325.460193,4185.082962,0.0,0.0,4517.038069,12692.449788,5405.036904,10952.474473


In [29]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'USA LNG','LNG less RU and USA', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [30]:
from datetime import datetime
today = date.today()

In [31]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng_q.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\ugne.keliauskaite\AppData\Local\Temp\ipykernel_24848\2207928906.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\ugne.keliauskaite\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
